# TransformerLens Circuit Analysis

Activation patching experiments mimic econometric impulse–response analyses: shock part of the model, observe downstream effects, and attribute contributions.


## Learning goals

1. Run a baseline forward pass with TransformerLens.
2. Patch specific attention heads and record logit differences.
3. Plot “response functions” analogous to IRFs.


In [ ]:
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer
from transformer_lens.hook_points import HookPoint

sns.set_theme(style="darkgrid")
MODEL_NAME = "NeelNanda/circuit_toy_model"  # lightweight demo model
model = HookedTransformer.from_pretrained(MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
prompt = "Credible policy guidance stabilizes inflation expectations."

In [ ]:
tokens = model.to_tokens(prompt)
logits = model(tokens)
log_probs = logits.log_softmax(dim=-1)[0, -1]
log_probs.topk(5)


In [ ]:
def patch_head(layer: int, head: int, cache_source: dict) -> float:
    """Return logit difference when replacing a head with cached activations."""

    def hook_fn(value: torch.Tensor, hook: HookPoint):
        value[:, :, head, :] = cache_source[hook.name][:, :, head, :]
        return value

    logits_patched = model.run_with_hooks(tokens, fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook_fn)])
    return (logits_patched - logits)[0, -1].detach()


In [ ]:
clean_prompt = prompt
corrupted_prompt = "Credible policy guidance destabilizes inflation expectations."

clean_tokens = model.to_tokens(clean_prompt)
corrupted_tokens = model.to_tokens(corrupted_prompt)

_, clean_cache = model.run_with_cache(clean_tokens)
_, corrupted_cache = model.run_with_cache(corrupted_tokens)


In [ ]:
records = []
for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        def hook_fn(value, hook, layer=layer, head=head):
            value[:, :, head, :] = clean_cache[hook.name][:, :, head, :]
            return value

        logits_patched = model.run_with_hooks(corrupted_tokens,
                                             fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook_fn)])
        delta = (logits_patched - model(corrupted_tokens))[0, -1]
        records.append({"layer": layer, "head": head, "delta": float(delta.max().item())})

results = pd.DataFrame.from_records(records)
results.head()


In [ ]:
pivot = results.pivot(index="layer", columns="head", values="delta")
plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, cmap="coolwarm", center=0)
plt.title("Logit delta when patching attention heads (clean -> corrupted)")
plt.ylabel("Layer")
plt.xlabel("Head")
plt.tight_layout()
plt.savefig("../docs/diagrams/transformerlens_patch_heatmap.png", dpi=300)
plt.show()


## Econometric takeaways

- Treat the logit delta matrix like an impulse–response surface: a positive delta indicates that restoring the clean head counteracts the corruption.
- Heads with large deltas behave like **identified channels** in a structural model. Document them just as you would significant impulse responses.
- Extending the experiment to MLP blocks and residual streams yields a Oaxaca–Blinder style decomposition of the final logit.
